# Quand ça plante : lire une erreur PyTorch

Relier le message à un contrat précis, puis vérifier la correction.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

Les erreurs sont provoquées puis interceptées. Le cas de placement oppose meta et CPU ; il ne mesure pas un appareil CUDA.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#lire-erreurs-pytorch).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False


## Nous lisons le programme

**1. Faire apparaître le contrat**

La première erreur compare une entrée de quatre colonnes à une couche construite pour trois mesures. Le message donne ces dimensions.

**2. Distinguer les autres propriétés**

Le type de cible, le device et la présence du gradient décrivent des causes différentes. Le cas meta reste une démonstration CPU.

**3. Réparer puis vérifier**

Le dernier cas concerne une vue incompatible avec le stockage. Les assertions vérifient les cinq corrections sans cacher les erreurs précédentes.

In [2]:
import torch
from torch import nn

def montrer(nom, operation):
    try:
        operation()
    except (RuntimeError, AttributeError) as erreur:
        print(nom, "|", type(erreur).__name__, "|", str(erreur))

montrer("forme", lambda: nn.Linear(3, 2)(torch.ones(2, 4)))
montrer("type", lambda: nn.CrossEntropyLoss()(
    torch.tensor([[1., 2.]]), torch.tensor([1.])))
montrer("device", lambda: torch.ones(2) + torch.ones(2, device="meta"))
montrer("gradient", lambda: torch.tensor(2., requires_grad=True).grad.item())
montrer("vue", lambda: torch.arange(6).reshape(2, 3).t().view(6))

assert nn.Linear(3, 2)(torch.ones(2, 3)).shape == (2, 2)
assert torch.isfinite(nn.CrossEntropyLoss()(
    torch.tensor([[1., 2.]]), torch.tensor([1], dtype=torch.long)))
assert torch.equal(torch.ones(2) + torch.ones(2), torch.tensor([2., 2.]))
w = torch.tensor(2., requires_grad=True)
(w ** 2).backward()
assert w.grad.item() == 4
assert torch.arange(6).reshape(2, 3).t().reshape(6).shape == (6,)
print("Cinq corrections vérifiées.")

forme | RuntimeError | mat1 and mat2 shapes cannot be multiplied (2x4 and 3x2)
type | RuntimeError | expected scalar type Long but found Float


device | RuntimeError | Tensor on device meta is not on the expected device cpu!
gradient | AttributeError | 'NoneType' object has no attribute 'item'
vue | RuntimeError | view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
Cinq corrections vérifiées.


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

Lire le message avant de modifier les données.

Distinguer forme, type, device, gradient et stockage.

Tester une réparation sur un exemple dont le résultat reste prévisible.

## Exercice 1 — Réparer le contrat de CrossEntropyLoss

Le modèle produit deux scores par exemple. Complétons le type et la forme des cibles.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
scores = torch.tensor([[1., 2.], [3., 1.]])
cibles = torch.tensor([1, 0], dtype=torch....)
perte = torch.nn.CrossEntropyLoss()(scores, cibles)
assert cibles.shape == (...)
assert torch.isfinite(perte)
```

**Piste :** Pour les cibles entières de cet exemple, la perte attend un indice long par ligne.

## Exercice 2 — Aplatir une transposée sans supposer sa contiguïté

Nous voulons les valeurs dans l’ordre logique du tenseur transposé. Complétons une opération adaptée.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
x = torch.arange(6).reshape(2, 3).t()
plat = x....(6)
assert plat.tolist() == [...]
```

**Piste :** reshape peut copier si nécessaire. Il ne transforme pas une transposition en l’ordre d’origine.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Réparer le contrat de CrossEntropyLoss</summary>

<pre><code>import torch
scores_exo = torch.tensor([[1., 2.], [3., 1.]])
cibles_exo = torch.tensor([1, 0], dtype=torch.long)
perte_exo = torch.nn.CrossEntropyLoss()(scores_exo, cibles_exo)
assert cibles_exo.shape == (2,)
assert torch.isfinite(perte_exo)
print(&#x27;cibles :&#x27;, cibles_exo.dtype, tuple(cibles_exo.shape))</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Aplatir une transposée sans supposer sa contiguïté</summary>

<pre><code>import torch
x_exo = torch.arange(6).reshape(2, 3).t()
plat_exo = x_exo.reshape(6)
assert plat_exo.tolist() == [0, 3, 1, 4, 2, 5]
print(&#x27;ordre après transposition :&#x27;, plat_exo.tolist())</code></pre>

</details>

